# Svenska kraftnat (SVK) – Linked Bids

This notebook demonstrates the two types of linked bids supported by SVK:

1. **Technically linked bids** (`linkedBidsIdentification`) – groups simple bids across
   consecutive MTUs under a shared UUID so the AOF cannot activate the same resource
   simultaneously in back-to-back quarters (double-activation guard).

2. **Conditionally linked bids** (`Linked_BidTimeSeries`) – adjusts a bid's availability
   in QH0 based on whether a linked bid in QH-1 or QH-2 was activated.

Both are commonly used by SVK BSPs operating hydro and thermal resources that have
ramp-time or capacity constraints spanning multiple quarter-hours.

## SVK key facts

| Parameter | Value |
|---|---|
| Receiver EIC | `10X1001A1001A418` |
| Control area EIC | `10YSE-1--------K` |
| Minimum bid | 1 MW |
| Message cut-off | 6 minutes (messages older than 6 min silently dropped) |
| Heartbeat | Every 5 minutes (xx:02, xx:07, xx:12, …) |
| Missing heartbeat | Bids unavailable for upcoming 2 quarters |
| Resource coding scheme | NSE (Swedish national) or A01 (EIC) |
| Sender coding scheme | NSE, A01, or A10 |


## Imports

In [1]:
from nexa_mfrr_eam import (
    Bid,
    BidDocument,
    BiddingZone,
    Direction,
    MARIMode,
    MarketProductType,
    TechnicalLink,
    TSO,
)
from nexa_mfrr_eam.types import ConditionalStatus

# Synthetic SVK BSP party ID (NSE coding scheme)
BSP_PARTY_ID = "99999"
BSP_CODING_SCHEME = "NSE"

# Synthetic resource object
RESOURCE_ID = "ZZZ"
RESOURCE_CODING = "NSE"

## Part 1: Technically Linked Bids

A `TechnicalLink` groups simple bids across multiple MTUs under a single shared
`linkedBidsIdentification` UUID.  The AOF uses this to avoid double-activation:
if a bid is directly activated in one quarter, the linked bids in adjacent quarters
are treated as blocked for the duration of the ramp.

### Example: hydro unit bidding four consecutive quarters

The unit can be direct-activated in any of the four MTUs.  The technical link
ensures that if MTU 10:00 is direct-activated and the unit is still ramping into
10:15, the 10:15 bid is not activated simultaneously.

In [2]:
tech_link = (
    TechnicalLink(bidding_zone=BiddingZone.SE2)
    .resource(RESOURCE_ID, coding_scheme=RESOURCE_CODING)
    .add_mtu(
        mtu="2021-11-09T10:00Z",
        direction=Direction.UP,
        volume_mw=26,
        price_eur=41.33,
        product_type=MarketProductType.SCHEDULED_AND_DIRECT,
        divisible=False,  # Indivisible – no minimum_Quantity
    )
    .add_mtu(
        mtu="2021-11-09T10:15Z",
        direction=Direction.UP,
        volume_mw=26,
        price_eur=41.33,
        product_type=MarketProductType.SCHEDULED_AND_DIRECT,
        divisible=True,
        min_volume_mw=6,
    )
    .add_mtu(
        mtu="2021-11-09T10:30Z",
        direction=Direction.UP,
        volume_mw=26,
        price_eur=41.33,
        product_type=MarketProductType.SCHEDULED_AND_DIRECT,
        divisible=True,
        min_volume_mw=14,
    )
    .add_mtu(
        mtu="2021-11-09T10:45Z",
        direction=Direction.UP,
        volume_mw=26,
        price_eur=41.33,
        product_type=MarketProductType.SCHEDULED_AND_DIRECT,
        divisible=True,
        min_volume_mw=14,
    )
    .build()
)

print(f"Link group ID:    {tech_link[0].linked_bids_identification}")
print(f"Number of bids:   {len(tech_link)}")
for bid in tech_link:
    print(f"  MTU {bid.period.time_interval_start.strftime('%H:%M')} – "
          f"{bid.period.point.quantity} MW @ {bid.period.point.energy_price} EUR/MWh "
          f"({'divisible' if bid.divisible_code == 'A01' else 'indivisible'})")

Link group ID:    6b3fc389-99fb-40c7-8a05-473852a9740c
Number of bids:   4
  MTU 10:00 – 26 MW @ 41.33 EUR/MWh (indivisible)
  MTU 10:15 – 26 MW @ 41.33 EUR/MWh (divisible)
  MTU 10:30 – 26 MW @ 41.33 EUR/MWh (divisible)
  MTU 10:45 – 26 MW @ 41.33 EUR/MWh (divisible)


### Wrap in a document and validate

In [3]:
doc_tech = (
    BidDocument(tso=TSO.SVK)
    .sender(party_id=BSP_PARTY_ID, coding_scheme=BSP_CODING_SCHEME)
    .add_bids(tech_link)
    .build()
)

errors = doc_tech.validate(mari_mode=MARIMode.PRE_MARI)
if errors:
    for e in errors:
        print(f"ERROR: {e}")
else:
    print(f"Document valid: {doc_tech.time_series_count} bids, receiver 10X1001A1001A418")

Document valid: 4 bids, receiver 10X1001A1001A418


### Preview XML output

In [4]:
xml_tech = doc_tech.to_xml().decode("utf-8")
print(xml_tech[:2500])

<?xml version='1.0' encoding='UTF-8'?>
<ReserveBid_MarketDocument xmlns="urn:iec62325.351:tc57wg16:451-7:reservebiddocument:7:4">
  <mRID>c5c34ac8-ae59-46e7-8721-9dd203978c2e</mRID>
  <revisionNumber>1</revisionNumber>
  <type>A37</type>
  <process.processType>A47</process.processType>
  <sender_MarketParticipant.mRID codingScheme="NSE">99999</sender_MarketParticipant.mRID>
  <sender_MarketParticipant.marketRole.type>A46</sender_MarketParticipant.marketRole.type>
  <receiver_MarketParticipant.mRID codingScheme="A01">10X1001A1001A418</receiver_MarketParticipant.mRID>
  <receiver_MarketParticipant.marketRole.type>A34</receiver_MarketParticipant.marketRole.type>
  <createdDateTime>2026-03-25T14:13:48Z</createdDateTime>
  <reserveBid_Period.timeInterval>
    <start>2021-11-09T10:00Z</start>
    <end>2021-11-09T11:00Z</end>
  </reserveBid_Period.timeInterval>
  <domain.mRID codingScheme="A01">10YSE-1--------K</domain.mRID>
  <subject_MarketParticipant.mRID codingScheme="NSE">99999</subject_

### Technically linked bids with duration constraints

SVK supports `maximum_ConstraintDuration.duration` and `resting_ConstraintDuration.duration`
for resources that can only run for a limited time and then need a rest period.
Set these once on the `TechnicalLink` builder and they apply to all MTUs.

In [5]:
constrained_link = (
    TechnicalLink(bidding_zone=BiddingZone.SE3)
    .resource(RESOURCE_ID, coding_scheme=RESOURCE_CODING)
    .max_duration(minutes=15)   # Maximum_ConstraintDuration.duration = PT15M
    .resting_time(minutes=30)   # Resting_ConstraintDuration.duration = PT30M
    .add_mtu(
        mtu="2021-11-09T10:00Z",
        direction=Direction.UP,
        volume_mw=50,
        price_eur=60.00,
        product_type=MarketProductType.SCHEDULED_AND_DIRECT,
        divisible=True,
        min_volume_mw=10,
    )
    .add_mtu(
        mtu="2021-11-09T10:15Z",
        direction=Direction.UP,
        volume_mw=50,
        price_eur=60.00,
        product_type=MarketProductType.SCHEDULED_AND_DIRECT,
        divisible=True,
        min_volume_mw=10,
    )
    .build()
)

for bid in constrained_link:
    print(f"MTU {bid.period.time_interval_start.strftime('%H:%M')}: "
          f"max={bid.maximum_constraint_duration}, "
          f"resting={bid.resting_constraint_duration}")

MTU 10:00: max=PT15M, resting=PT30M
MTU 10:15: max=PT15M, resting=PT30M


## Part 2: Conditionally Linked Bids

Conditional links allow a bid's availability in QH0 to depend on the activation
outcome of a linked bid in QH-1 or QH-2.  This is useful when:

- A resource cannot be activated twice in quick succession (e.g. thermal unit
  already ramping in QH-1 should not bid in QH0).
- A resource should only be available in QH0 *if* it was activated in QH-1
  (e.g. a battery that must be pre-conditioned).

### Condition status codes

| Code | Parent bid status | Meaning |
|------|-------------------|---------|
| `A55` | A65 (conditionally available) | If linked bid **activated** → this bid becomes **unavailable** |
| `A56` | A65 (conditionally available) | If linked bid **not activated** → this bid becomes **unavailable** |
| `A67` | A66 (conditionally unavailable) | If linked bid **activated** → this bid becomes **available** |

### Example from SVK reference XML

Three bids across three consecutive MTUs:

- **Bid 1** (22:15 QH): Normal available bid (A06).
- **Bid 2** (22:30 QH): Conditionally *unavailable* (A66) – becomes available only
  if Bid 1 was activated (A67 condition).
- **Bid 3** (22:45 QH): Conditionally *available* (A65) – becomes unavailable if
  Bid 1 was activated (A55) AND becomes unavailable if Bid 2 was *not* activated (A56).

In [6]:
# Bid 1 – normal available bid in QH0 (22:15)
bid_qh0 = (
    Bid.up(volume_mw=71, price_eur=93.57)
    .divisible(min_volume_mw=30)
    .for_mtu("2022-02-03T22:15Z")
    .resource(RESOURCE_ID, coding_scheme=RESOURCE_CODING)
    .bidding_zone(BiddingZone.SE1)
    .product_type(MarketProductType.SCHEDULED_AND_DIRECT)
    .build()
)

# Bid 2 – conditionally unavailable (A66) in QH+1 (22:30)
# Becomes available only if bid_qh0 was activated (A67)
bid_qh1 = (
    Bid.up(volume_mw=30, price_eur=45.06)
    .divisible(min_volume_mw=0)
    .for_mtu("2022-02-03T22:30Z")
    .resource(RESOURCE_ID, coding_scheme=RESOURCE_CODING)
    .bidding_zone(BiddingZone.SE1)
    .product_type(MarketProductType.SCHEDULED_ONLY)
    .conditionally_unavailable()                                    # status A66
    .link_to(bid_qh0, ConditionalStatus.AVAILABLE_IF_LINKED_ACTIVATED)  # A67
    .build()
)

# Bid 3 – conditionally available (A65) in QH+2 (22:45)
# Becomes unavailable if bid_qh0 was activated (A55)
# Becomes unavailable if bid_qh1 was not activated (A56)
bid_qh2 = (
    Bid.up(volume_mw=30, price_eur=45.06)
    .divisible(min_volume_mw=0)
    .for_mtu("2022-02-03T22:45Z")
    .resource(RESOURCE_ID, coding_scheme=RESOURCE_CODING)
    .bidding_zone(BiddingZone.SE1)
    .product_type(MarketProductType.SCHEDULED_AND_DIRECT)
    .conditionally_available()                                         # status A65
    .link_to(bid_qh0, ConditionalStatus.NOT_AVAILABLE_IF_ACTIVATED)    # A55
    .link_to(bid_qh1, ConditionalStatus.AVAILABLE_IF_ACTIVATED)        # A56
    .build()
)

print(f"Bid 1 ({bid_qh0.period.time_interval_start.strftime('%H:%M')}): status={bid_qh0.status_value}, links={len(bid_qh0.linked_bid_time_series)}")
print(f"Bid 2 ({bid_qh1.period.time_interval_start.strftime('%H:%M')}): status={bid_qh1.status_value}, links={len(bid_qh1.linked_bid_time_series)}")
for lk in bid_qh1.linked_bid_time_series:
    print(f"    → links to {lk.mrid[:8]}... condition={lk.status_value}")
print(f"Bid 3 ({bid_qh2.period.time_interval_start.strftime('%H:%M')}): status={bid_qh2.status_value}, links={len(bid_qh2.linked_bid_time_series)}")
for lk in bid_qh2.linked_bid_time_series:
    print(f"    → links to {lk.mrid[:8]}... condition={lk.status_value}")

Bid 1 (22:15): status=A06, links=0
Bid 2 (22:30): status=A66, links=1
    → links to 471f13f0... condition=A67
Bid 3 (22:45): status=A65, links=2
    → links to 471f13f0... condition=A55
    → links to 7ac680ed... condition=A56


### Wrap in a document and validate

In [7]:
doc_cond = (
    BidDocument(tso=TSO.SVK)
    .sender(party_id=BSP_PARTY_ID, coding_scheme=BSP_CODING_SCHEME)
    .add_bid(bid_qh0)
    .add_bid(bid_qh1)
    .add_bid(bid_qh2)
    .build()
)

errors = doc_cond.validate(mari_mode=MARIMode.PRE_MARI)
if errors:
    for e in errors:
        print(f"ERROR: {e}")
else:
    print(f"Document valid: {doc_cond.time_series_count} bids")

Document valid: 3 bids


### Preview XML output

Notice that the `Linked_BidTimeSeries` elements appear inside the relevant
`Bid_TimeSeries` elements, after the `Period` and `Reason` sections.

In [8]:
xml_cond = doc_cond.to_xml().decode("utf-8")
print(xml_cond[:2000])

<?xml version='1.0' encoding='UTF-8'?>
<ReserveBid_MarketDocument xmlns="urn:iec62325.351:tc57wg16:451-7:reservebiddocument:7:4">
  <mRID>caa7b0b3-5b7f-4697-966d-18e787d861c6</mRID>
  <revisionNumber>1</revisionNumber>
  <type>A37</type>
  <process.processType>A47</process.processType>
  <sender_MarketParticipant.mRID codingScheme="NSE">99999</sender_MarketParticipant.mRID>
  <sender_MarketParticipant.marketRole.type>A46</sender_MarketParticipant.marketRole.type>
  <receiver_MarketParticipant.mRID codingScheme="A01">10X1001A1001A418</receiver_MarketParticipant.mRID>
  <receiver_MarketParticipant.marketRole.type>A34</receiver_MarketParticipant.marketRole.type>
  <createdDateTime>2026-03-25T14:13:49Z</createdDateTime>
  <reserveBid_Period.timeInterval>
    <start>2022-02-03T22:15Z</start>
    <end>2022-02-03T23:00Z</end>
  </reserveBid_Period.timeInterval>
  <domain.mRID codingScheme="A01">10YSE-1--------K</domain.mRID>
  <subject_MarketParticipant.mRID codingScheme="NSE">99999</subject_

## Part 3: Mixed document – technical + conditional links

A real BSP document may contain a mix of independently linked bids and conditionally
linked bids.  The library handles both in the same `BidDocument`.

In [9]:
# Technical link for a different resource (SE3 bidding zone)
tech_bids = (
    TechnicalLink(bidding_zone=BiddingZone.SE3)
    .resource("YYY", coding_scheme="NSE")
    .add_mtu(
        mtu="2022-02-03T22:15Z",
        direction=Direction.UP,
        volume_mw=40,
        price_eur=55.00,
        product_type=MarketProductType.SCHEDULED_AND_DIRECT,
        divisible=True,
        min_volume_mw=10,
    )
    .add_mtu(
        mtu="2022-02-03T22:30Z",
        direction=Direction.UP,
        volume_mw=40,
        price_eur=55.00,
        product_type=MarketProductType.SCHEDULED_AND_DIRECT,
        divisible=True,
        min_volume_mw=10,
    )
    .build()
)

doc_mixed = (
    BidDocument(tso=TSO.SVK)
    .sender(party_id=BSP_PARTY_ID, coding_scheme=BSP_CODING_SCHEME)
    .add_bids(tech_bids)          # 2 technically linked bids
    .add_bid(bid_qh0)             # 1 normal bid
    .add_bid(bid_qh1)             # 1 conditionally unavailable bid
    .add_bid(bid_qh2)             # 1 conditionally available bid
    .build()
)

errors = doc_mixed.validate(mari_mode=MARIMode.PRE_MARI)
print(f"Total bids: {doc_mixed.time_series_count}")
print(f"Validation errors: {len(errors)}")
if errors:
    for e in errors:
        print(f"  {e}")

Total bids: 5
Validation errors: 0


## Summary

| Feature | Method / Class | XML element |
|---------|---------------|-------------|
| Technical link group | `TechnicalLink(...).add_mtu(...).build()` | `<linkedBidsIdentification>` |
| Conditionally available | `.conditionally_available()` on `SimpleBidBuilder` | `<status><value>A65</value></status>` |
| Conditionally unavailable | `.conditionally_unavailable()` on `SimpleBidBuilder` | `<status><value>A66</value></status>` |
| Conditional link condition | `.link_to(bid, ConditionalStatus.X)` | `<Linked_BidTimeSeries>` |
| Max activation duration | `.max_duration(minutes=N)` | `<maximum_ConstraintDuration.duration>` |
| Resting time | `.resting_time(minutes=N)` | `<resting_ConstraintDuration.duration>` |